In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from sklearn.metrics import roc_auc_score

PROCESSED = Path("../data/processed")
REPORTS = Path("reports")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

In [2]:
bb = pd.read_parquet(PROCESSED / "bureau_balance.parquet")
print(f"Shape: {bb.shape}")
print(f"Memoria: {bb.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nDtypes:\n{bb.dtypes}")
print(f"\nHead:")
print(bb.head())

Shape: (27299925, 3)
Memoria: 1431.9 MB

Dtypes:
SK_ID_BUREAU       int32
MONTHS_BALANCE      int8
STATUS            object
dtype: object

Head:
   SK_ID_BUREAU  MONTHS_BALANCE STATUS
0       5715448               0      C
1       5715448              -1      C
2       5715448              -2      C
3       5715448              -3      C
4       5715448              -4      C


In [3]:
print(f"SK_ID_BUREAU únicos: {bb['SK_ID_BUREAU'].nunique():,}")
print(f"MONTHS_BALANCE rango: [{bb['MONTHS_BALANCE'].min()}, {bb['MONTHS_BALANCE'].max()}]")
print(f"  → cubre {abs(bb['MONTHS_BALANCE'].min())} meses de historial mensual")

print(f"\n=== STATUS ===")
sv = bb["STATUS"].value_counts(dropna=False)
sp = (sv / len(bb) * 100).round(3)
print(pd.DataFrame({"count": sv, "pct": sp}))

months_per_b = bb.groupby("SK_ID_BUREAU").size()
print(f"\n=== Meses observados por crédito ===")
print(months_per_b.describe().round(1))

SK_ID_BUREAU únicos: 817,395
MONTHS_BALANCE rango: [-96, 0]
  → cubre 96 meses de historial mensual

=== STATUS ===
           count     pct
STATUS                  
C       13646993  49.989
0        7499507  27.471
X        5810482  21.284
1         242347   0.888
5          62406   0.229
2          23419   0.086
3           8924   0.033
4           5847   0.021

=== Meses observados por crédito ===
count    817395.0
mean         33.4
std          25.8
min           1.0
25%          13.0
50%          26.0
75%          48.0
max          97.0
dtype: float64


In [4]:
dpd_map = {"C": 0, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
bb["DPD_LEVEL"] = bb["STATUS"].map(dpd_map)
bb["HAS_DPD"] = (bb["DPD_LEVEL"].fillna(0) > 0).astype("int8")

print("=== DPD_LEVEL (C→0, X→NaN) ===")
print(bb["DPD_LEVEL"].value_counts(dropna=False).sort_index())
print(f"\nFilas con cualquier DPD: {bb['HAS_DPD'].sum():,} ({bb['HAS_DPD'].mean()*100:.2f}%)")

=== DPD_LEVEL (C→0, X→NaN) ===
DPD_LEVEL
0.0    21146500
1.0      242347
2.0       23419
3.0        8924
4.0        5847
5.0       62406
NaN     5810482
Name: count, dtype: int64

Filas con cualquier DPD: 342,943 (1.26%)


In [5]:
# Conteos por status
status_counts = pd.crosstab(bb["SK_ID_BUREAU"], bb["STATUS"])
status_counts.columns = [f"BB_STATUS_{c}_CNT" for c in status_counts.columns]

# Proporciones
status_pct = status_counts.div(status_counts.sum(axis=1), axis=0).astype("float32")
status_pct.columns = [c.replace("_CNT", "_PCT") for c in status_pct.columns]

# Estadísticos de DPD y meses
dpd_stats = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_OBSERVED=("MONTHS_BALANCE", "count"),
    BB_MONTHS_OLDEST=("MONTHS_BALANCE", "min"),
    BB_DPD_MAX=("DPD_LEVEL", "max"),
    BB_DPD_MEAN=("DPD_LEVEL", "mean"),
    BB_HAS_DPD_ANY=("HAS_DPD", "max"),
    BB_HAS_DPD_SUM=("HAS_DPD", "sum"),
)

bb_at_bureau = dpd_stats.join(status_counts).join(status_pct)
print(f"Shape a nivel SK_ID_BUREAU: {bb_at_bureau.shape}")
print(bb_at_bureau.head(3))

del status_counts, status_pct, dpd_stats
gc.collect()

Shape a nivel SK_ID_BUREAU: (817395, 22)
              BB_MONTHS_OBSERVED  BB_MONTHS_OLDEST  BB_DPD_MAX  BB_DPD_MEAN  \
SK_ID_BUREAU                                                                  
5001709                       97               -96         0.0          0.0   
5001710                       83               -82         0.0          0.0   
5001711                        4                -3         0.0          0.0   

              BB_HAS_DPD_ANY  BB_HAS_DPD_SUM  BB_STATUS_0_CNT  \
SK_ID_BUREAU                                                    
5001709                    0               0                0   
5001710                    0               0                5   
5001711                    0               0                3   

              BB_STATUS_1_CNT  BB_STATUS_2_CNT  BB_STATUS_3_CNT  \
SK_ID_BUREAU                                                      
5001709                     0                0                0   
5001710                     0       

29

In [6]:
bureau_map = pd.read_parquet(
    PROCESSED / "bureau.parquet",
    columns=["SK_ID_BUREAU", "SK_ID_CURR"]
)

bb_at_bureau = bb_at_bureau.reset_index().merge(bureau_map, on="SK_ID_BUREAU", how="left")
n_unmapped = bb_at_bureau["SK_ID_CURR"].isna().sum()
print(f"SK_ID_BUREAU sin mapeo a cliente: {n_unmapped:,} "
      f"({100*n_unmapped/len(bb_at_bureau):.2f}%)")

bb_at_bureau = bb_at_bureau.dropna(subset=["SK_ID_CURR"])
print(f"Shape después de filtrar: {bb_at_bureau.shape}")

SK_ID_BUREAU sin mapeo a cliente: 43,041 (5.27%)
Shape después de filtrar: (774354, 24)


In [7]:
status_cols = [c for c in bb_at_bureau.columns if c.startswith("BB_STATUS_")]

agg_dict = {
    "BB_MONTHS_OBSERVED": ["mean", "sum"],
    "BB_DPD_MAX": ["max", "mean"],
    "BB_DPD_MEAN": "mean",
    "BB_HAS_DPD_ANY": ["max", "sum"],
    "BB_HAS_DPD_SUM": "sum",
}
for c in status_cols:
    agg_dict[c] = "mean"   # promedio entre créditos del cliente

bb_at_client = bb_at_bureau.groupby("SK_ID_CURR").agg(agg_dict)
bb_at_client.columns = ["_".join(c).upper() if isinstance(c, tuple) else c
                        for c in bb_at_client.columns]
bb_at_client = bb_at_client.reset_index()

print(f"Shape a nivel cliente: {bb_at_client.shape}")
print(bb_at_client.head(3))

Shape a nivel cliente: (134542, 25)
   SK_ID_CURR  BB_MONTHS_OBSERVED_MEAN  BB_MONTHS_OBSERVED_SUM  \
0    100001.0                24.571429                     172   
1    100002.0                13.750000                     110   
2    100005.0                 7.000000                      21   

   BB_DPD_MAX_MAX  BB_DPD_MAX_MEAN  BB_DPD_MEAN_MEAN  BB_HAS_DPD_ANY_MAX  \
0             1.0         0.142857          0.010989                   1   
1             1.0         0.750000          0.299222                   1   
2             0.0         0.000000          0.000000                   0   

   BB_HAS_DPD_ANY_SUM  BB_HAS_DPD_SUM_SUM  BB_STATUS_0_CNT_MEAN  \
0                   1                   1              4.428571   
1                   6                  27              5.625000   
2                   0                   0              4.666667   

   BB_STATUS_1_CNT_MEAN  BB_STATUS_2_CNT_MEAN  BB_STATUS_3_CNT_MEAN  \
0              0.142857                   0.0         

In [8]:
train_target = pd.read_parquet(
    PROCESSED / "application_train_reduced.parquet",
    columns=["SK_ID_CURR", "TARGET"]
)
bb_target = bb_at_client.merge(train_target, on="SK_ID_CURR", how="inner")
print(f"Clientes con bureau_balance + target: {bb_target.shape}")

bb_features = [c for c in bb_target.columns if c.startswith("BB_")]
auc_list = []

for col in bb_features:
    m = bb_target[col].notna()
    
    if m.sum() == 0 or bb_target.loc[m, col].nunique() < 2:
        continue
        
    auc_raw = roc_auc_score(bb_target.loc[m, "TARGET"], bb_target.loc[m, col])
    
    auc_list.append({
        "feature": col,
        "auc": round(max(auc_raw, 1 - auc_raw), 4),
        "direction": "↓ menos default" if auc_raw < 0.5 else "↑ más default",
        "pct_null": round(100 * bb_target[col].isna().mean(), 2),
    })

auc_bb = pd.DataFrame(auc_list).sort_values("auc", ascending=False)
print(f"\n=== Top 20 features de bureau_balance por AUC ===")
print(auc_bb.head(20).to_string(index=False))
auc_bb.to_csv(REPORTS / "auc_rank_bureau_balance_agg.csv", index=False)

Clientes con bureau_balance + target: (92231, 26)

=== Top 20 features de bureau_balance por AUC ===
                feature    auc       direction  pct_null
BB_MONTHS_OBSERVED_MEAN 0.5952 ↓ menos default      0.00
   BB_STATUS_C_CNT_MEAN 0.5751 ↓ menos default      0.00
   BB_STATUS_C_PCT_MEAN 0.5600 ↓ menos default      0.00
 BB_MONTHS_OBSERVED_SUM 0.5575 ↓ menos default      0.00
       BB_DPD_MEAN_MEAN 0.5509   ↑ más default      1.75
   BB_STATUS_1_PCT_MEAN 0.5503   ↑ más default      0.00
   BB_STATUS_0_PCT_MEAN 0.5483   ↑ más default      0.00
        BB_DPD_MAX_MEAN 0.5452   ↑ más default      1.75
   BB_STATUS_1_CNT_MEAN 0.5448   ↑ más default      0.00
     BB_HAS_DPD_SUM_SUM 0.5433   ↑ más default      0.00
     BB_HAS_DPD_ANY_SUM 0.5427   ↑ más default      0.00
   BB_STATUS_0_CNT_MEAN 0.5412 ↓ menos default      0.00
         BB_DPD_MAX_MAX 0.5410   ↑ más default      1.75
     BB_HAS_DPD_ANY_MAX 0.5408   ↑ más default      0.00
   BB_STATUS_X_CNT_MEAN 0.5279 ↓ menos defau

In [9]:
bb_at_client.to_parquet(PROCESSED / "bureau_balance_aggregated.parquet", index=False)
mem = bb_at_client.memory_usage(deep=True).sum() / 1024**2
print(f"Guardado: bureau_balance_aggregated.parquet")
print(f"Shape: {bb_at_client.shape} | Memoria: {mem:.1f} MB")

Guardado: bureau_balance_aggregated.parquet
Shape: (134542, 25) | Memoria: 19.8 MB


In [10]:
bureau_agg = pd.read_parquet(PROCESSED / "bureau_aggregated.parquet")
bureau_master = bureau_agg.merge(bb_at_client, on="SK_ID_CURR", how="left")

# Clientes con bureau pero sin bureau_balance
n_no_bb = bureau_master["BB_MONTHS_OBSERVED_MEAN"].isna().sum()
print(f"Clientes con bureau pero sin bureau_balance: {n_no_bb:,} "
      f"({100*n_no_bb/len(bureau_master):.2f}%)")

bureau_master.to_parquet(PROCESSED / "bureau_master.parquet", index=False)
print(f"\nbureau_master guardado: {bureau_master.shape}")
print(f"Memoria: {bureau_master.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

del bb, bb_at_bureau, bb_target, bureau_agg, bureau_master
gc.collect()

Clientes con bureau pero sin bureau_balance: 171,269 (56.00%)

bureau_master guardado: (305811, 49)
Memoria: 91.9 MB


0